<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 38px; border-radius: 10px; color: #0f172a; font-family: sans-serif;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Pipeline Stage 08
    </span>
    <h1 style="color: #0f172a; margin-top: 10px; margin-bottom: 8px; font-size: 26px; border-bottom: none;">
        Covariance Construction
    </h1>
    <p style="color: #475569; font-size: 14px; margin-bottom: 20px;">
        Builds the 20-day covariance matrix &Sigma; the optimizer uses to measure portfolio risk (w<sup>T</sup>&Sigma;w). It combines the per-asset volatility forecasts from notebook <code>07_walk_forward_baseline_model.ipynb</code> with a correlation matrix estimated from trailing daily returns: risk levels come from the validated volatility model, co-movement from history.
    </p>
    <div style="background-color: #f1f5f9; padding: 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: bold; text-transform: uppercase;">Key Objectives in this Module:</p>
        <ol style="margin-top: 8px; margin-bottom: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li><b>Calendar Alignment:</b> Every asset on one common calendar without losing any price move &mdash; crypto weekend and holiday returns are folded into the next common trading day.</li>
            <li><b>Shrunk Correlation:</b> Correlation of the market assets over the trailing 252 days, shrunk with Ledoit-Wolf so the optimizer cannot exploit estimation noise.</li>
            <li><b>Synthetic Assets:</b> Bond and RealEstate get zero correlation with every other asset, because notebook 03 generates them from independent random shocks; their sample correlations are checked against pure noise.</li>
            <li><b>&Sigma; = D&middot;R&middot;D:</b> The correlation matrix scaled by the 20-day predicted volatilities from notebook 07, so &Sigma; is on the same horizon as <code>Expected_Return_20D</code>.</li>
            <li><b>Validation &amp; Save:</b> Symmetry, positive definiteness and an exact diagonal are asserted, then &Sigma; is saved in the optimizer's ticker order.</li>
        </ol>
    </div>
    <p style="margin-top: 15px; margin-bottom: 0; color: #64748b; font-size: 12px;">
        <b>Next Milestones:</b> NSGA-II Optimization
    </p>
</div>

In [ ]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
from src.config import FEATURE_DIR, DATA_DIR
from src.helpers.charts import chart_dir
from src.helpers.asof import cut_to_as_of
from src.risk.covariance import (align_returns, shrunk_correlation, embed_synthetic, build_covariance,
                                 synthetic_correlation_check, validate_covariance, class_portfolio_vol_from_sigma)
from src.risk.covariance_plots import plot_correlation_heatmap, plot_eigenvalue_spectrum

CHART_DIR = chart_dir(DATA_DIR, "covariance_construction")
WINDOW = 252                                # trailing days for correlation, same window as the class betas in notebook 07
HORIZON = 20                                # matches Expected_Return_20D and Predicted_Volatility_20D
SYNTHETIC_CLASSES = ["Bond", "RealEstate"]  # GBM series from notebook 03 with independent shocks

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 01
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Load Optimizer Inputs &amp; Align Return Calendars
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Loads the live optimizer inputs from notebook <code>07_walk_forward_baseline_model.ipynb</code> and the daily log returns from the post-EDA feature file, then builds one return matrix in which every row covers the same calendar interval for every asset.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>The optimizer's asset order is fixed here:</b> &Sigma; rows and columns follow the ticker order in <code>optimizer_inputs.parquet</code>, so the weight vector and &Sigma; can never be misaligned.</li>
            <li><b>Why alignment is needed:</b> Markets close on different holidays, so some dates have no return for one or more assets; dropping those dates would delete their price moves, while summing log returns into the next common date keeps every move.</li>
            <li><b>Guards:</b> one live date, features ending on that same date, and a full 252-day window with no gaps &mdash; any failure stops the run.</li>
        </ul>
    </div>
</div>

In [ ]:
opt = pd.read_parquet(DATA_DIR / "interim" / "optimizer_inputs.parquet")
assert opt["Ticker"].is_unique and opt["Date"].nunique() == 1, "optimizer_inputs must hold one row per ticker on one date"
LIVE_DATE = pd.Timestamp(opt["Date"].iloc[0])

features = cut_to_as_of(pd.read_parquet(FEATURE_DIR / "clean_features_after_EDA.parquet"), LIVE_DATE)
assert pd.Timestamp(features["Date"].max()) == LIVE_DATE

tickers = opt["Ticker"].tolist()                      # the optimizer's asset order
classes = opt.set_index("Ticker")["Asset_Class"]
synthetic = classes.index[classes.isin(SYNTHETIC_CLASSES)].tolist()
market = [t for t in tickers if t not in synthetic]

aligned, n_folded = align_returns(features, market)
returns = aligned.iloc[-WINDOW:]
assert len(returns) == WINDOW, f"only {len(returns)} common dates, need {WINDOW}"
assert pd.Timestamp(returns.index[-1]) == LIVE_DATE, "correlation window must end on the live date"
assert returns.notna().all().all(), "gaps left after alignment"

print(f"{len(tickers)} tickers: {len(market)} market, {len(synthetic)} synthetic ({', '.join(synthetic)})")
print(f"dates folded into the next common date (full history): {n_folded}")
print(f"correlation window: {returns.index[0]:%Y-%m-%d} to {returns.index[-1]:%Y-%m-%d} ({len(returns)} common dates)")

<!-- ===== STEP 01 KEY TAKEAWAY  (replaces the markdown cell after the Step 01 code cell) ===== -->
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Alignment Output
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Every market asset now has a return on the same 252 common dates, ending on the live date.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Folded dates:</b> 59 over the full history, about 6% of the roughly 1,000 days. That fits holidays that differ across markets; weekend crypto rows would fold several hundred dates. Their returns were moved into the next common date, so no price move is lost.</li>
            <li><b>Window:</b> 252 common dates ending on the live date &mdash; the same horizon as the class betas in notebook 07.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 02
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Shrunk Correlation of the Market Assets
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Estimates the correlation matrix of the market assets from the aligned window and shrinks it with Ledoit-Wolf.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>Why shrink:</b> with about these many assets and 252 days, the sample correlation is noisy and its smallest eigenvalues are biased toward zero &mdash; exactly the directions an optimizer loads up on, because they look almost riskless. Shrinkage pulls the matrix toward the identity by the amount that minimizes expected estimation error, computed from the data, so there is nothing to tune.</li>
            <li><b>Why correlation, not covariance:</b> returns are standardized first, so Crypto's large variance cannot dominate the shrinkage. Volatility levels come from the notebook 07 model instead (Step 04).</li>
            <li>The eigenvalue chart is saved to <code>../covariance_construction/&lt;run date&gt;/eigenvalue_spectrum.png</code>.</li>
        </ul>
    </div>
</div>

In [ ]:
sample_corr, market_corr, shrinkage = shrunk_correlation(returns)
print(f"Ledoit-Wolf shrinkage intensity: {shrinkage:.3f}  ({len(market)} assets, {len(returns)} days)")

ev = pd.DataFrame({"Sample": np.linalg.eigvalsh(sample_corr.to_numpy()),
                   "Shrunk": np.linalg.eigvalsh(market_corr.to_numpy())})
display(pd.DataFrame({"Smallest_Eigenvalue": ev.min(), "Largest_Eigenvalue": ev.max(),
                      "Condition_Number": ev.max() / ev.min()}).round(3))

plot_eigenvalue_spectrum(sample_corr, market_corr, save_dir=CHART_DIR)



<!-- ===== STEP 02 KEY TAKEAWAY  (replaces the markdown cell after the Step 02 code cell) ===== -->
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Shrinkage Result
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        A moderate correction: Ledoit-Wolf put about 13% of the weight on the identity matrix, enough to lift the noisiest directions without touching the main risk factors.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Shrinkage intensity:</b> 0.129 &mdash; 13% of the weight on the identity matrix and 87% on the sample correlation. For these many assets and 252 days that is a moderate correction: the sample matrix is mostly reliable, but its tail is noisy.</li>
            <li><b>Conditioning:</b> the smallest eigenvalue rises from 0.051 to 0.174 (3.4&times;) and the condition number falls from 178.4 to 46.6 (3.8&times; lower).</li>
            <li><b>What is preserved:</b> every eigenvalue is pulled toward 1 by the same weight, so the largest (the common market factor) moves only from 9.15 to 8.10 (&minus;11.5%). The curves in the chart coincide for roughly the first 20 ranks and separate only in the tail &mdash; the market and class factors are kept, and only the noisy low-variance directions an optimizer would over-exploit are lifted.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 03
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Synthetic Assets: Zero Correlation by Construction
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Bond and RealEstate are generated in notebook 03 from a fixed drift plus independent random shocks (Bond is fully deterministic). Their true correlation with every other asset is therefore exactly zero, and any sample correlation is noise.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>Check:</b> over the same 252-day window, each synthetic asset's sample correlation with every market asset is compared with the &plusmn;2/&radic;T band that pure noise stays inside about 95% of the time.</li>
            <li><b>Bond:</b> its daily return changes only with the number of calendar days since the previous row (a Monday carries three days of accrual), so any correlation is a calendar artifact, not risk.</li>
            <li><b>Decision:</b> both assets enter &Sigma; with zero off-diagonal entries, consistent with notebook 07 using their known parameters instead of estimates.</li>
        </ul>
    </div>
</div>

In [ ]:
syn_check = synthetic_correlation_check(features, synthetic, market, window=WINDOW)
display(syn_check.round(3))


<!-- ===== STEP 03 KEY TAKEAWAY  (replaces the markdown cell after the Step 03 code cell) ===== -->
<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Synthetic Check
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Zero correlation is the right choice for both: RealEstate is indistinguishable from noise, and Bond's deviations are too small to matter.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>RealEstate:</b> mean correlation 0.025, largest 0.145, and 6.4% of pairs outside the &plusmn;0.126 band &mdash; in line with the ~5% that pure noise produces.</li>
            <li><b>Bond:</b> mean correlation 0.054, largest 0.198, and 21.3% of pairs outside the band, above the ~5% expected from noise. Bond has no random component, so any correlation most likely comes from its calendar pattern (more accrual on Mondays and after holidays), and all the market correlations share a common market factor, so they move together rather than independently. The size is what matters: a mean of 0.054 is negligible.</li>
            <li><b>Consequence:</b> zero is the right value for both. RealEstate's shocks are independent by construction, and Bond's volatility is only 0.14%, so even a correlation of 0.2 would change its covariance with a 20%-volatility asset by about 0.1% of that asset's own variance.</li>
        </ul>
    </div>
</div>




<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 04
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Build &Sigma; = D&middot;R&middot;D and Validate
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Places the shrunk market correlation and the synthetic assets into one correlation matrix R in the optimizer's ticker order, then scales it by D, the 20-day predicted volatilities from notebook 07 (Bond and RealEstate use their known volatilities from notebook 07, Step 06).
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>Units:</b> &Sigma; is a 20-day covariance, the same horizon as <code>Expected_Return_20D</code>, so return and risk are comparable inside the optimizer.</li>
            <li><b>Validation:</b> &Sigma; must be symmetric and positive definite, its diagonal must reproduce the predicted variances exactly, and the synthetic off-diagonals must be exactly zero &mdash; any failure stops the run.</li>
            <li><b>Cross-check:</b> the annualized volatility of each class's equal-weighted portfolio implied by &Sigma; is compared with <code>Class_Portfolio_Vol</code> in notebook 07, Step 07. Small differences are expected because notebook 07 used unshrunk correlations.</li>
            <li>The correlation heatmap, grouped by class, is saved to <code>../covariance_construction/&lt;run date&gt;/correlation_heatmap.png</code>.</li>
        </ul>
    </div>
</div>

In [ ]:
corr = embed_synthetic(market_corr, tickers)
vol_20d = opt.set_index("Ticker")["Predicted_Volatility_20D"]
sigma = build_covariance(corr, vol_20d)

checks = validate_covariance(sigma, corr, vol_20d, synthetic)
display(checks.to_frame("Value"))
assert checks["Max_Asymmetry"] < 1e-12, "Sigma is not symmetric"
assert checks["Min_Eigenvalue_Sigma"] > 0, "Sigma is not positive definite"
assert checks["Max_Diag_Error"] < 1e-12, "Sigma diagonal does not match predicted variances"
assert checks["Max_Abs_Corr_Synthetic"] == 0, "synthetic assets must have zero correlation"

display(class_portfolio_vol_from_sigma(sigma, classes, horizon=HORIZON).to_frame().round(4))
plot_correlation_heatmap(corr, classes, save_dir=CHART_DIR)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Validation
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        &Sigma; passed every guard and is a valid covariance matrix: w<sup>T</sup>&Sigma;w is positive for any non-zero weight vector.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Correlation condition number:</b> 46.6, identical to the shrunk market matrix in Step 02 because the synthetic assets add only unit eigenvalues (178.4 unshrunk). &Sigma;'s own smallest eigenvalue, 1.6&times;10<sup>&minus;7</sup>, is simply Bond's 20-day variance (0.14% annual volatility) &mdash; a scale effect, not instability, which is why the condition number is read on the correlation matrix.</li>
            <li><b>Class portfolio volatility:</b> the equal-weighted class portfolios implied by &Sigma; have annual volatilities of Crypto 40.2%, Commodity 26.1%, Stocks 14.2%, RealEstate 12.5%, ETF 12.0% and Bond 0.14%. Bond and RealEstate are single-asset classes, so their values are exactly the volatilities fixed for them in notebook 07, Step 06.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 05
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Save &Sigma; for the Optimizer
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Saves &Sigma; and R to <code>DATA_DIR/interim/</code>, next to <code>optimizer_inputs.parquet</code>, both indexed by <code>Ticker</code> in the optimizer's order.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li>The saved file is reloaded and compared with the in-memory matrix, so what the optimizer reads is exactly what was validated.</li>
        </ul>
    </div>
</div>

In [ ]:
out_dir = DATA_DIR / "interim"
sigma.to_parquet(out_dir / "covariance_20d.parquet")
corr.to_parquet(out_dir / "correlation_matrix.parquet")

reloaded = pd.read_parquet(out_dir / "covariance_20d.parquet")
assert reloaded.index.tolist() == tickers and reloaded.columns.tolist() == tickers, "saved order differs from optimizer_inputs"
assert np.array_equal(reloaded.to_numpy(), sigma.to_numpy()), "saved matrix differs from validated matrix"
print(f"saved {sigma.shape[0]}x{sigma.shape[1]} covariance (20-day) and correlation to {out_dir}")